# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mass spectrometry protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets and their fields (all referenced by `@id`).

Below, we print record sets with their fields and corresponding columns, all by their `@id`s.

In [ ]:
# View all record sets and their schema
rs_dict = {}
print("Record Sets in the dataset:\n")
for rs in dataset.record_sets():
    print(f"  @id: {getattr(rs, '@id', '')} | name: {getattr(rs, 'name', '')}")
    fields = getattr(rs, 'fields', [])
    if fields:
        print(f"    Fields:")
        for field in fields:
            print(f"      @id: {getattr(field, '@id', '')} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'dataType', '')}")
            columns = getattr(field, 'columns', [])
            if columns:
                print(f"        Columns:")
                for col in columns:
                    print(f"          @id: {getattr(col, '@id', '')} | header: {getattr(col, 'header', '')}")
    rs_dict[getattr(rs, '@id', '')] = rs

print("\nExample record from each record set by @id:\n")
for rs_id in rs_dict:
    print(f"- Record set @id: {rs_id}")
    # Only load first record for quick overview
    gen = dataset.records(record_set=rs_id)
    for i, rec in enumerate(gen):
        print(rec)
        break
    print()

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. All references are by their `@id` field.

In [ ]:
# Extract all record set @id's from the dataset
record_sets = [rs_id for rs_id in rs_dict]
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded record set {record_set} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load data for record set {record_set}: {e}")

# Preview columns and first rows for largest/main record set
if len(dataframes) > 0:
    largest_rs = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nColumns in record set {largest_rs}:\n", dataframes[largest_rs].columns.tolist())
    display(dataframes[largest_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing values, grouping, etc. All column references are by their `@id` (which also serve as DataFrame columns).

In [ ]:
# For demonstration, use numeric fields from the main (largest) record set
record_set_id = largest_rs
df = dataframes[record_set_id]
print(f"Working on record set @id: {record_set_id}")

# Identify numeric fields (int or float)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    # Try to convert floats
    for c in df.columns:
        # Some numeric columns may be loaded as strings; Try convert
        df[c] = pd.to_numeric(df[c], errors='ignore')
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Detected numeric fields: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Pick the first numeric field
    print(f"Using numeric field '@id': {numeric_field} for filtering and normalization.")
    threshold = df[numeric_field].mean()  # Example threshold at mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} rows")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Find a candidate field for grouping: choose the first object (non-numeric)
    group_field = None
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]) and c != numeric_field:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable string field for grouping found.")
else:
    print("No numeric fields found in record set for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        # Visualize group means
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print('No numeric fields to visualize.')

## 6. Conclusion
This notebook demonstrated loading, inspecting, preprocessing, and visualizing the FAIR² dataset using the Croissant schema and the `mlcroissant` library.

You can expand the EDA and visualization sections to further analyze the proteins and features referenced in this dataset. All fields and record sets are referenced by their `@id` to ensure alignment with the Croissant standard.